In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/btc_usdt_1h_clean.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print("Shape:", df.shape)
df.head()

Shape: (32125, 6)


,timestamp,open,high,low,close,volume
0,2023-01-01 00:00:00,16541.77,16545.70,16508.39,16529.67,4364.83570
1,2023-01-01 01:00:00,16529.59,16556.80,16525.78,16551.47,3590.06669
2,2023-01-01 02:00:00,16551.47,16559.77,16538.14,16548.19,3318.84038
3,2023-01-01 03:00:00,16548.19,16548.19,16518.21,16533.04,4242.08050
4,2023-01-01 04:00:00,16533.04,16535.97,16511.92,16521.85,4285.00909


In [2]:
# Price returns over different horizons
df["return_1h"] = df["close"].pct_change(1)
df["return_3h"] = df["close"].pct_change(3)
df["return_6h"] = df["close"].pct_change(6)
df["return_24h"] = df["close"].pct_change(24)

df[["timestamp", "close", "return_1h", "return_3h", "return_6h", "return_24h"]].tail(10)

,timestamp,close,return_1h,return_3h,return_6h,return_24h
32115,2026-08-31 03:00:00,77756.71,0.003649,-0.002314,-0.011395,-0.004965
32116,2026-08-31 04:00:00,77622.28,-0.001729,-0.000240,-0.010198,-0.006241
32117,2026-08-31 05:00:00,78016.00,0.005072,0.006996,0.004300,-0.001430
32118,2026-08-31 06:00:00,78093.87,0.000998,0.004336,0.002012,-0.002382
32119,2026-08-31 07:00:00,78196.01,0.001308,0.007391,0.007149,0.000999
32120,2026-08-31 08:00:00,78480.80,0.003642,0.005958,0.012995,0.005210
32121,2026-08-31 09:00:00,78511.03,0.000385,0.005342,0.009701,0.006552
32122,2026-08-31 10:00:00,78710.01,0.002534,0.006573,0.014013,0.007227
32123,2026-08-31 11:00:00,78315.43,-0.005013,-0.002107,0.003838,0.001912
32124,2026-08-31 12:00:00,78416.00,0.001284,-0.001210,0.004125,-0.004684


In [3]:
# Exponential Moving Averages
df["ema20"] = df["close"].ewm(span=20, adjust=False).mean()
df["ema50"] = df["close"].ewm(span=50, adjust=False).mean()
df["ema200"] = df["close"].ewm(span=200, adjust=False).mean()

# Distance of price from each EMA (as a %) — shows trend strength
df["ema20_dist"] = (df["close"] - df["ema20"]) / df["ema20"]
df["ema50_dist"] = (df["close"] - df["ema50"]) / df["ema50"]
df["ema200_dist"] = (df["close"] - df["ema200"]) / df["ema200"]

df[["timestamp", "close", "ema20", "ema50", "ema200", "ema20_dist", "ema50_dist", "ema200_dist"]].tail(10)


,timestamp,close,ema20,ema50,ema200,ema20_dist,ema50_dist,ema200_dist
32115,2026-08-31 03:00:00,77756.71,78220.493708,78306.306116,77340.378496,-0.005929,-0.007019,0.005383
32116,2026-08-31 04:00:00,77622.28,78163.520974,78279.481563,77343.183486,-0.006924,-0.008396,0.003609
32117,2026-08-31 05:00:00,78016.00,78149.471358,78269.148952,77349.878178,-0.001708,-0.003234,0.008612
32118,2026-08-31 06:00:00,78093.87,78144.175990,78262.275268,77357.281082,-0.000644,-0.002152,0.009522
32119,2026-08-31 07:00:00,78196.01,78149.112563,78259.676630,77365.626643,0.000600,-0.000814,0.010733
32120,2026-08-31 08:00:00,78480.80,78180.701842,78268.348135,77376.722895,0.003839,0.002714,0.014269
32121,2026-08-31 09:00:00,78511.03,78212.161667,78277.865071,77388.009533,0.003821,0.002979,0.014512
32122,2026-08-31 10:00:00,78710.01,78259.575794,78294.811931,77401.163767,0.005756,0.005303,0.016910
32123,2026-08-31 11:00:00,78315.43,78264.895242,78295.620482,77410.260943,0.000646,0.000253,0.011693
32124,2026-08-31 12:00:00,78416.00,78279.286171,78300.341248,77420.268297,0.001746,0.001477,0.012861


In [4]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)

    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df["rsi"] = compute_rsi(df["close"], period=14)

df[["timestamp", "close", "rsi"]].tail(10)

,timestamp,close,rsi
32115,2026-08-31 03:00:00,77756.71,40.979386
32116,2026-08-31 04:00:00,77622.28,39.050091
32117,2026-08-31 05:00:00,78016.00,46.930586
32118,2026-08-31 06:00:00,78093.87,48.352894
32119,2026-08-31 07:00:00,78196.01,50.236838
32120,2026-08-31 08:00:00,78480.80,55.149355
32121,2026-08-31 09:00:00,78511.03,55.649838
32122,2026-08-31 10:00:00,78710.01,58.900788
32123,2026-08-31 11:00:00,78315.43,50.928448
32124,2026-08-31 12:00:00,78416.00,52.686251


In [5]:
def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

df["macd"], df["macd_signal"], df["macd_hist"] = compute_macd(df["close"])

df[["timestamp", "close", "macd", "macd_signal", "macd_hist"]].tail(10)

,timestamp,close,macd,macd_signal,macd_hist
32115,2026-08-31 03:00:00,77756.71,-133.966438,16.882807,-150.849245
32116,2026-08-31 04:00:00,77622.28,-163.032046,-19.100163,-143.931883
32117,2026-08-31 05:00:00,78016.00,-152.538468,-45.787824,-106.750644
32118,2026-08-31 06:00:00,78093.87,-136.366822,-63.903624,-72.463198
32119,2026-08-31 07:00:00,78196.01,-113.994771,-73.921853,-40.072918
32120,2026-08-31 08:00:00,78480.80,-72.449434,-73.627370,1.177935
32121,2026-08-31 09:00:00,78511.03,-36.662525,-66.234401,29.571876
32122,2026-08-31 10:00:00,78710.01,7.666507,-51.454219,59.120726
32123,2026-08-31 11:00:00,78315.43,10.833396,-38.996696,49.830092
32124,2026-08-31 12:00:00,78416.00,21.213797,-26.954597,48.168394


In [6]:
# Bollinger Bands (20-period, 2 standard deviations)
bb_period = 20
df["bb_middle"] = df["close"].rolling(bb_period).mean()
bb_std = df["close"].rolling(bb_period).std()
df["bb_upper"] = df["bb_middle"] + (2 * bb_std)
df["bb_lower"] = df["bb_middle"] - (2 * bb_std)
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["bb_middle"]

# ATR (Average True Range, 14-period)
high_low = df["high"] - df["low"]
high_close_prev = (df["high"] - df["close"].shift(1)).abs()
low_close_prev = (df["low"] - df["close"].shift(1)).abs()
true_range = pd.concat([high_low, high_close_prev, low_close_prev], axis=1).max(axis=1)
df["atr"] = true_range.ewm(alpha=1/14, min_periods=14, adjust=False).mean()

df[["timestamp", "close", "bb_upper", "bb_middle", "bb_lower", "bb_width", "atr"]].tail(10)

,timestamp,close,bb_upper,bb_middle,bb_lower,bb_width,atr
32115,2026-08-31 03:00:00,77756.71,79491.543206,78402.6380,77313.732794,0.027777,459.902622
32116,2026-08-31 04:00:00,77622.28,79515.407076,78380.0515,77244.695924,0.028971,449.215292
32117,2026-08-31 05:00:00,78016.00,79515.101740,78380.8510,77246.600260,0.028942,463.041342
32118,2026-08-31 06:00:00,78093.87,79515.009327,78378.2800,77241.550673,0.029006,450.867675
32119,2026-08-31 07:00:00,78196.01,79515.408678,78379.7805,77244.152322,0.028978,439.273555
32120,2026-08-31 08:00:00,78480.80,79485.391324,78364.5675,77243.743676,0.028605,446.460444
32121,2026-08-31 09:00:00,78511.03,79446.121189,78347.1190,77248.116811,0.028055,443.545413
32122,2026-08-31 10:00:00,78710.01,79436.751905,78343.4340,77250.116095,0.027911,441.512883
32123,2026-08-31 11:00:00,78315.43,79387.718188,78318.0240,77248.329812,0.027317,439.120534
32124,2026-08-31 12:00:00,78416.00,79230.222056,78271.9275,77313.632944,0.024486,426.887639


In [7]:
# Volume change (% change hour to hour)
df["volume_change"] = df["volume"].pct_change(1)

# Volume relative to its own rolling average (z-score style ratio)
volume_ma20 = df["volume"].rolling(20).mean()
df["volume_ratio"] = df["volume"] / volume_ma20

df[["timestamp", "volume", "volume_change", "volume_ratio"]].tail(10)

,timestamp,volume,volume_change,volume_ratio
32115,2026-08-31 03:00:00,278.92536,-0.513558,0.576463
32116,2026-08-31 04:00:00,550.02773,0.971953,1.090948
32117,2026-08-31 05:00:00,511.92091,-0.069282,0.981619
32118,2026-08-31 06:00:00,968.02210,0.890960,1.724016
32119,2026-08-31 07:00:00,716.03378,-0.260313,1.216313
32120,2026-08-31 08:00:00,782.74546,0.093168,1.333186
32121,2026-08-31 09:00:00,624.76686,-0.201826,1.051936
32122,2026-08-31 10:00:00,576.71834,-0.076906,0.959932
32123,2026-08-31 11:00:00,482.35572,-0.163620,0.807672
32124,2026-08-31 12:00:00,177.62604,-0.631753,0.310144


In [8]:
# Rolling volatility (standard deviation of 1h returns, 24h window)
df["volatility_24h"] = df["return_1h"].rolling(24).std()

df[["timestamp", "close", "return_1h", "volatility_24h"]].tail(10)

,timestamp,close,return_1h,volatility_24h
32115,2026-08-31 03:00:00,77756.71,0.003649,0.003568
32116,2026-08-31 04:00:00,77622.28,-0.001729,0.003582
32117,2026-08-31 05:00:00,78016.00,0.005072,0.003743
32118,2026-08-31 06:00:00,78093.87,0.000998,0.003726
32119,2026-08-31 07:00:00,78196.01,0.001308,0.003711
32120,2026-08-31 08:00:00,78480.80,0.003642,0.003780
32121,2026-08-31 09:00:00,78511.03,0.000385,0.003772
32122,2026-08-31 10:00:00,78710.01,0.002534,0.003787
32123,2026-08-31 11:00:00,78315.43,-0.005013,0.003939
32124,2026-08-31 12:00:00,78416.00,0.001284,0.003582


In [9]:
print("All columns:")
print(df.columns.tolist())

print("\nShape:", df.shape)

print("\nNaN count per column:")
print(df.isnull().sum())

print("\nFirst valid (non-NaN) row index for each column:")
print(df.apply(lambda col: col.first_valid_index()))

All columns:
['timestamp', 'open', 'high', 'low', 'close', 'volume', 'return_1h', 'return_3h', 'return_6h', 'return_24h', 'ema20', 'ema50', 'ema200', 'ema20_dist', 'ema50_dist', 'ema200_dist', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_middle', 'bb_upper', 'bb_lower', 'bb_width', 'atr', 'volume_change', 'volume_ratio', 'volatility_24h']

Shape: (32125, 28)

NaN count per column:
timestamp          0
open               0
high               0
low                0
close              0
volume             0
return_1h          1
return_3h          3
return_6h          6
return_24h        24
ema20              0
ema50              0
ema200             0
ema20_dist         0
ema50_dist         0
ema200_dist        0
rsi               13
macd               0
macd_signal        0
macd_hist          0
bb_middle         19
bb_upper          19
bb_lower          19
bb_width          19
atr               13
volume_change      1
volume_ratio      19
volatility_24h    24
dtype: int64

First valid 

In [10]:
# Drop rows where any feature is still NaN (only affects the first ~24 rows)
before_len = len(df)
df = df.dropna().reset_index(drop=True)
after_len = len(df)

print(f"Dropped {before_len - after_len} rows")
print(f"Remaining rows: {after_len}")
print("\nAny NaNs left?", df.isnull().values.any())

Dropped 24 rows
Remaining rows: 32101

Any NaNs left? False


In [11]:
output_path = "../data/processed/btc_usdt_1h_features.csv"
df.to_csv(output_path, index=False)
print(f"Saved feature-engineered dataset: {len(df)} rows, {len(df.columns)} columns")
print(f"Saved to: {output_path}")

Saved feature-engineered dataset: 32101 rows, 28 columns
Saved to: ../data/processed/btc_usdt_1h_features.csv


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/btc_usdt_1h_clean.csv",
    parse_dates=["timestamp"]
)

df = df.sort_values("timestamp").reset_index(drop=True)

print("Shape:", df.shape)
print(df.head())

Shape: (32125, 6)
            timestamp      open      high       low     close      volume
0 2023-01-01 00:00:00  16541.77  16545.70  16508.39  16529.67  4364.83570
1 2023-01-01 01:00:00  16529.59  16556.80  16525.78  16551.47  3590.06669
2 2023-01-01 02:00:00  16551.47  16559.77  16538.14  16548.19  3318.84038
3 2023-01-01 03:00:00  16548.19  16548.19  16518.21  16533.04  4242.08050
4 2023-01-01 04:00:00  16533.04  16535.97  16511.92  16521.85  4285.00909


In [2]:
# Future 6-hour return
df["future_return_6h"] = df["close"].shift(-6) / df["close"] - 1

# Create trading signal
df["target"] = np.select(
    [
        df["future_return_6h"] > 0.005,
        df["future_return_6h"] < -0.005
    ],
    [
        "BUY",
        "SELL"
    ],
    default="HOLD"
)

df[["timestamp", "close", "future_return_6h", "target"]].tail(15)

,timestamp,close,future_return_6h,target
32110,2026-08-30 22:00:00,78422.01,-0.010198,SELL
32111,2026-08-30 23:00:00,77682.00,0.004300,HOLD
32112,2026-08-31 00:00:00,77937.04,0.002012,HOLD
32113,2026-08-31 01:00:00,77640.94,0.007149,BUY
32114,2026-08-31 02:00:00,77473.99,0.012995,BUY
32115,2026-08-31 03:00:00,77756.71,0.009701,BUY
32116,2026-08-31 04:00:00,77622.28,0.014013,BUY
32117,2026-08-31 05:00:00,78016.00,0.003838,HOLD
32118,2026-08-31 06:00:00,78093.87,0.004125,HOLD
32119,2026-08-31 07:00:00,78196.01,NaN,HOLD


In [3]:
# Remove rows where a 6-hour future return cannot be calculated
before_len = len(df)

df = df.dropna(subset=["future_return_6h"]).reset_index(drop=True)

after_len = len(df)

print(f"Dropped {before_len - after_len} rows")
print(f"Remaining rows: {after_len}")
print("\nTarget distribution:")
print(df["target"].value_counts())

Dropped 6 rows
Remaining rows: 32119

Target distribution:
target
HOLD    16323
BUY      8224
SELL     7572
Name: count, dtype: int64


In [4]:
print("Target counts:")
print(df["target"].value_counts())

print("\nTarget percentages:")
print((df["target"].value_counts(normalize=True) * 100).round(2))

print("\nMissing values:")
print(df[["future_return_6h", "target"]].isnull().sum())

print("\nFuture return statistics:")
print(df["future_return_6h"].describe())

Target counts:
target
HOLD    16323
BUY      8224
SELL     7572
Name: count, dtype: int64

Target percentages:
target
HOLD    50.82
BUY     25.60
SELL    23.57
Name: proportion, dtype: float64

Missing values:
future_return_6h    0
target              0
dtype: int64

Future return statistics:
count    32119.000000
mean         0.000362
std          0.011967
min         -0.099317
25%         -0.004598
50%          0.000205
75%          0.005210
max          0.108364
Name: future_return_6h, dtype: float64


In [5]:
output_path = "../data/processed/btc_usdt_1h_ml.csv"

df.to_csv(output_path, index=False)

print(f"Saved ML-ready dataset: {len(df)} rows, {len(df.columns)} columns")
print(f"Saved to: {output_path}")
print("\nColumns:")
print(df.columns.tolist())

Saved ML-ready dataset: 32119 rows, 8 columns
Saved to: ../data/processed/btc_usdt_1h_ml.csv

Columns:
['timestamp', 'open', 'high', 'low', 'close', 'volume', 'future_return_6h', 'target']


In [6]:
import pandas as pd
import numpy as np

# Load the feature-engineered dataset
features_path = "../data/processed/btc_usdt_1h_features.csv"

df = pd.read_csv(
    features_path,
    parse_dates=["timestamp"]
)

df = df.sort_values("timestamp").reset_index(drop=True)

print("Feature dataset loaded:")
print("Shape:", df.shape)

# Calculate future 6-hour return
df["future_return_6h"] = df["close"].shift(-6) / df["close"] - 1

# Create BUY / HOLD / SELL target
df["target"] = np.select(
    [
        df["future_return_6h"] > 0.005,
        df["future_return_6h"] < -0.005
    ],
    [
        "BUY",
        "SELL"
    ],
    default="HOLD"
)

# Remove final 6 rows where future return cannot be calculated
df = df.dropna(subset=["future_return_6h"]).reset_index(drop=True)

# Save correct ML-ready dataset
output_path = "../data/processed/btc_usdt_1h_ml.csv"

df.to_csv(output_path, index=False)

print("\nML-ready dataset saved successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Saved to:", output_path)

print("\nTarget distribution:")
print(df["target"].value_counts())

print("\nTechnical features included:")
print(df.columns.tolist())

Feature dataset loaded:
Shape: (32101, 28)

ML-ready dataset saved successfully
Rows: 32095
Columns: 30
Saved to: ../data/processed/btc_usdt_1h_ml.csv

Target distribution:
target
HOLD    16299
BUY      8224
SELL     7572
Name: count, dtype: int64

Technical features included:
['timestamp', 'open', 'high', 'low', 'close', 'volume', 'return_1h', 'return_3h', 'return_6h', 'return_24h', 'ema20', 'ema50', 'ema200', 'ema20_dist', 'ema50_dist', 'ema200_dist', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_middle', 'bb_upper', 'bb_lower', 'bb_width', 'atr', 'volume_change', 'volume_ratio', 'volatility_24h', 'future_return_6h', 'target']


In [7]:
import pandas as pd
import numpy as np

# Load final ML-ready dataset
df = pd.read_csv(
    "../data/processed/btc_usdt_1h_ml.csv",
    parse_dates=["timestamp"]
)

# Make sure data is chronological
df = df.sort_values("timestamp").reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Start:", df["timestamp"].min())
print("End:", df["timestamp"].max())
print("\nTarget distribution:")
print(df["target"].value_counts())
print("\nMissing values:", df.isnull().sum().sum())

Dataset shape: (32095, 30)
Start: 2023-01-02 00:00:00
End: 2026-08-31 06:00:00

Target distribution:
target
HOLD    16299
BUY      8224
SELL     7572
Name: count, dtype: int64

Missing values: 0


In [8]:
# Chronological 80/20 train-test split

split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("Total rows:", len(df))
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining period:")
print(train_df["timestamp"].min(), "to", train_df["timestamp"].max())

print("\nTesting period:")
print(test_df["timestamp"].min(), "to", test_df["timestamp"].max())

print("\nTrain target distribution:")
print(train_df["target"].value_counts(normalize=True).mul(100).round(2))

print("\nTest target distribution:")
print(test_df["target"].value_counts(normalize=True).mul(100).round(2))

Total rows: 32095
Training rows: 25676
Testing rows: 6419

Training period:
2023-01-02 00:00:00 to 2025-12-06 19:00:00

Testing period:
2025-12-06 20:00:00 to 2026-08-31 06:00:00

Train target distribution:
target
HOLD    50.99
BUY     25.81
SELL    23.20
Name: proportion, dtype: float64

Test target distribution:
target
HOLD    49.95
SELL    25.16
BUY     24.89
Name: proportion, dtype: float64


In [10]:
import numpy as np

# Check every numeric column for inf values
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = {}

for col in numeric_cols:
    n_inf = np.isinf(df[col]).sum()
    if n_inf > 0:
        inf_counts[col] = n_inf

print("Columns with infinite values:")
print(inf_counts)

Columns with infinite values:
{'volume_change': np.int64(1)}


In [11]:
# Locate the row(s) with infinite volume_change
inf_rows = df[np.isinf(df["volume_change"])]
print(inf_rows[["timestamp", "volume", "volume_change"]])

# Also check the row right before it (the "previous volume" that caused the divide)
idx = inf_rows.index[0]
print("\nContext around the bad row:")
print(df.loc[idx-2:idx+2, ["timestamp", "volume", "volume_change"]])

               timestamp      volume  volume_change
1957 2023-03-24 13:00:00  4491.62009            inf

Context around the bad row:
               timestamp      volume  volume_change
1955 2023-03-24 11:00:00  1267.41714      -0.651736
1956 2023-03-24 12:00:00     0.00000      -1.000000
1957 2023-03-24 13:00:00  4491.62009            inf
1958 2023-03-24 14:00:00  8983.24018       1.000000
1959 2023-03-24 15:00:00  5198.28681      -0.421335


In [12]:
# Replace inf/-inf with NaN, then drop any row that has one (only affects the 1 known row)
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

train_mask = X_train.notnull().all(axis=1)
test_mask = X_test.notnull().all(axis=1)

print(f"Dropping {(~train_mask).sum()} train rows, {(~test_mask).sum()} test rows due to inf/NaN")

X_train, y_train = X_train[train_mask], y_train[train_mask]
X_test, y_test = X_test[test_mask], y_test[test_mask]

Dropping 1 train rows, 0 test rows due to inf/NaN


In [13]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred, labels=["BUY", "HOLD", "SELL"]))

Classification Report:

              precision    recall  f1-score   support

         BUY       1.00      1.00      1.00      1598
        HOLD       1.00      1.00      1.00      3206
        SELL       1.00      1.00      1.00      1615

    accuracy                           1.00      6419
   macro avg       1.00      1.00      1.00      6419
weighted avg       1.00      1.00      1.00      6419

Confusion Matrix (rows=actual, cols=predicted):
[[1598    0    0]
 [   0 3205    1]
 [   0    7 1608]]


In [14]:
feature_cols = [c for c in df.columns if c not in 
                 ["timestamp", "open", "high", "low", "close", "volume", 
                  "future_return_6h", "target"]]

X_train = train_df[feature_cols]
y_train = train_df["target"]
X_test = test_df[feature_cols]
y_test = test_df["target"]

# Re-apply the inf cleanup since we rebuilt X_train/X_test
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
train_mask = X_train.notnull().all(axis=1)
test_mask = X_test.notnull().all(axis=1)
X_train, y_train = X_train[train_mask], y_train[train_mask]
X_test, y_test = X_test[test_mask], y_test[test_mask]

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred, labels=["BUY", "HOLD", "SELL"]))

Classification Report:

              precision    recall  f1-score   support

         BUY       0.34      0.16      0.22      1598
        HOLD       0.52      0.91      0.67      3206
        SELL       0.38      0.02      0.05      1615

    accuracy                           0.50      6419
   macro avg       0.42      0.37      0.31      6419
weighted avg       0.44      0.50      0.40      6419

Confusion Matrix (rows=actual, cols=predicted):
[[ 259 1300   39]
 [ 262 2917   27]
 [ 234 1341   40]]


In [15]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred, labels=["BUY", "HOLD", "SELL"]))

Classification Report:

              precision    recall  f1-score   support

         BUY       0.31      0.35      0.33      1598
        HOLD       0.64      0.42      0.50      3206
        SELL       0.28      0.45      0.35      1615

    accuracy                           0.41      6419
   macro avg       0.41      0.40      0.39      6419
weighted avg       0.47      0.41      0.42      6419

Confusion Matrix (rows=actual, cols=predicted):
[[ 555  388  655]
 [ 698 1336 1172]
 [ 525  371  719]]


In [16]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

volatility_24h    0.138158
bb_width          0.105693
atr               0.094267
volume_ratio      0.072072
ema20             0.047882
ema50             0.047501
bb_middle         0.045426
bb_upper          0.045385
ema200            0.044635
ema200_dist       0.044300
return_24h        0.042596
bb_lower          0.039053
macd              0.031094
ema50_dist        0.030643
ema20_dist        0.030381
macd_signal       0.029138
rsi               0.024160
return_1h         0.023309
macd_hist         0.020326
return_3h         0.018991
return_6h         0.018735
volume_change     0.006256
dtype: float64


In [17]:
# Simple binary target: did price go up or down over the next 6 hours (no HOLD band)
df["target_binary"] = np.where(df["future_return_6h"] > 0, "UP", "DOWN")

print(df["target_binary"].value_counts(normalize=True).mul(100).round(2))

target_binary
UP      51.29
DOWN    48.71
Name: proportion, dtype: float64


In [18]:
# Rebuild train/test split (same chronological 80/20 split as before)
train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df[feature_cols]
y_train = train_df["target_binary"]
X_test = test_df[feature_cols]
y_test = test_df["target_binary"]

# Same inf cleanup as before
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
train_mask = X_train.notnull().all(axis=1)
test_mask = X_test.notnull().all(axis=1)
X_train, y_train = X_train[train_mask], y_train[train_mask]
X_test, y_test = X_test[test_mask], y_test[test_mask]

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred, labels=["UP", "DOWN"]))

Classification Report:

              precision    recall  f1-score   support

        DOWN       0.50      0.40      0.45      3171
          UP       0.51      0.61      0.56      3248

    accuracy                           0.51      6419
   macro avg       0.51      0.51      0.50      6419
weighted avg       0.51      0.51      0.50      6419

Confusion Matrix (rows=actual, cols=predicted):
[[1979 1269]
 [1889 1282]]


In [19]:
# Longer horizon: 24-hour future return instead of 6-hour
df["future_return_24h"] = df["close"].shift(-24) / df["close"] - 1

df["target_24h"] = np.where(df["future_return_24h"] > 0, "UP", "DOWN")

# Drop rows where 24h future return can't be calculated (last 24 rows)
before_len = len(df)
df = df.dropna(subset=["future_return_24h"]).reset_index(drop=True)
after_len = len(df)

print(f"Dropped {before_len - after_len} rows")
print(f"Remaining rows: {after_len}")
print("\nTarget distribution:")
print(df["target_24h"].value_counts(normalize=True).mul(100).round(2))

Dropped 24 rows
Remaining rows: 32071

Target distribution:
target_24h
UP      52.26
DOWN    47.74
Name: proportion, dtype: float64


In [20]:
# Rebuild split index since row count changed
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df[feature_cols]
y_train = train_df["target_24h"]
X_test = test_df[feature_cols]
y_test = test_df["target_24h"]

# Same inf cleanup as before
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
train_mask = X_train.notnull().all(axis=1)
test_mask = X_test.notnull().all(axis=1)
X_train, y_train = X_train[train_mask], y_train[train_mask]
X_test, y_test = X_test[test_mask], y_test[test_mask]

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred, labels=["UP", "DOWN"]))

Classification Report:

              precision    recall  f1-score   support

        DOWN       0.44      0.28      0.34      3163
          UP       0.48      0.65      0.55      3252

    accuracy                           0.47      6415
   macro avg       0.46      0.46      0.45      6415
weighted avg       0.46      0.47      0.45      6415

Confusion Matrix (rows=actual, cols=predicted):
[[2118 1134]
 [2287  876]]


In [21]:
# Future volatility (24h ahead) vs current volatility
horizon = 6  # predict volatility regime 6 hours ahead

df["future_volatility"] = df["volatility_24h"].shift(-horizon)

# Regime target: is future volatility higher or lower than current?
df["vol_target"] = np.where(df["future_volatility"] > df["volatility_24h"], "EXPAND", "CONTRACT")

# Drop rows where future volatility can't be calculated
before_len = len(df)
df = df.dropna(subset=["future_volatility"]).reset_index(drop=True)
after_len = len(df)

print(f"Dropped {before_len - after_len} rows")
print(f"Remaining rows: {after_len}")
print("\nTarget distribution:")
print(df["vol_target"].value_counts(normalize=True).mul(100).round(2))

Dropped 6 rows
Remaining rows: 32065

Target distribution:
vol_target
EXPAND      50.02
CONTRACT    49.98
Name: proportion, dtype: float64


In [22]:
# Rebuild split index since row count changed again
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

# Exclude future_volatility and vol_target from features (and future_return columns, which are leaks)
feature_cols = [c for c in df.columns if c not in 
                 ["timestamp", "open", "high", "low", "close", "volume", 
                  "future_return_6h", "target", "target_binary",
                  "future_return_24h", "target_24h",
                  "future_volatility", "vol_target"]]

X_train = train_df[feature_cols]
y_train = train_df["vol_target"]
X_test = test_df[feature_cols]
y_test = test_df["vol_target"]

# Same inf cleanup as before
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
train_mask = X_train.notnull().all(axis=1)
test_mask = X_test.notnull().all(axis=1)
X_train, y_train = X_train[train_mask], y_train[train_mask]
X_test, y_test = X_test[test_mask], y_test[test_mask]

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred, labels=["EXPAND", "CONTRACT"]))

Classification Report:

              precision    recall  f1-score   support

    CONTRACT       0.61      0.60      0.61      3242
      EXPAND       0.60      0.62      0.61      3171

    accuracy                           0.61      6413
   macro avg       0.61      0.61      0.61      6413
weighted avg       0.61      0.61      0.61      6413

Confusion Matrix (rows=actual, cols=predicted):
[[1953 1218]
 [1298 1944]]


In [23]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

volatility_24h    0.212106
volume_ratio      0.159060
rsi               0.130540
bb_width          0.086979
ema20_dist        0.068980
return_24h        0.034368
ema50_dist        0.034346
macd_hist         0.032508
return_6h         0.029340
macd              0.028452
atr               0.022817
return_3h         0.020682
macd_signal       0.020046
ema200_dist       0.018233
ema200            0.015206
ema20             0.013900
ema50             0.013796
bb_upper          0.013442
bb_middle         0.013321
bb_lower          0.012469
return_1h         0.010460
volume_change     0.008948
dtype: float64


In [24]:
import os
os.makedirs("../models", exist_ok=True)

In [25]:
import joblib

vol_dataset_path = "../data/processed/btc_usdt_1h_volatility.csv"
df.to_csv(vol_dataset_path, index=False)
print(f"Saved volatility dataset: {len(df)} rows, {len(df.columns)} columns")
print(f"Saved to: {vol_dataset_path}")

model_path = "../models/volatility_rf_baseline.pkl"
joblib.dump(model, model_path)
print(f"\nSaved model to: {model_path}")

feature_cols_path = "../models/volatility_rf_baseline_features.pkl"
joblib.dump(feature_cols, feature_cols_path)
print(f"Saved feature list to: {feature_cols_path}")

Saved volatility dataset: 32065 rows, 35 columns
Saved to: ../data/processed/btc_usdt_1h_volatility.csv

Saved model to: ../models/volatility_rf_baseline.pkl
Saved feature list to: ../models/volatility_rf_baseline_features.pkl


In [26]:
try:
    import xgboost as xgb
    print("XGBoost already installed, version:", xgb.__version__)
except ImportError:
    print("XGBoost not installed — run: pip install xgboost")

XGBoost already installed, version: 3.4.1


In [27]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# XGBoost needs numeric labels, not strings
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train_enc)
y_pred_enc = xgb_model.predict(X_test)
y_pred = le.inverse_transform(y_pred_enc)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred, labels=["EXPAND", "CONTRACT"]))

Classification Report:

              precision    recall  f1-score   support

    CONTRACT       0.66      0.56      0.61      3242
      EXPAND       0.61      0.71      0.66      3171

    accuracy                           0.63      6413
   macro avg       0.64      0.63      0.63      6413
weighted avg       0.64      0.63      0.63      6413

Confusion Matrix (rows=actual, cols=predicted):
[[2246  925]
 [1434 1808]]


In [28]:
xgb_importances = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(xgb_importances)

volume_ratio      0.112090
bb_width          0.081984
volatility_24h    0.080403
rsi               0.079736
ema20_dist        0.045523
bb_middle         0.041340
bb_lower          0.040144
bb_upper          0.039529
ema50             0.039209
atr               0.038678
return_24h        0.038391
ema200            0.038280
macd_hist         0.038054
macd              0.037851
macd_signal       0.035719
ema20             0.035435
ema50_dist        0.034026
ema200_dist       0.033707
return_3h         0.030768
return_6h         0.029534
return_1h         0.026237
volume_change     0.023362
dtype: float32


In [29]:
from sklearn.metrics import accuracy_score

n_splits = 5
total_len = len(df)
fold_size = total_len // (n_splits + 1)  # +1 because first fold is just training data

fold_results = []

for i in range(1, n_splits + 1):
    train_end = fold_size * i
    test_end = fold_size * (i + 1)
    
    fold_train = df.iloc[:train_end]
    fold_test = df.iloc[train_end:test_end]
    
    X_tr = fold_train[feature_cols].replace([np.inf, -np.inf], np.nan)
    y_tr = fold_train["vol_target"]
    X_te = fold_test[feature_cols].replace([np.inf, -np.inf], np.nan)
    y_te = fold_test["vol_target"]
    
    tr_mask = X_tr.notnull().all(axis=1)
    te_mask = X_te.notnull().all(axis=1)
    X_tr, y_tr = X_tr[tr_mask], y_tr[tr_mask]
    X_te, y_te = X_te[te_mask], y_te[te_mask]
    
    y_tr_enc = le.fit_transform(y_tr)
    y_te_enc = le.transform(y_te)
    
    fold_model = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, eval_metric="logloss"
    )
    fold_model.fit(X_tr, y_tr_enc)
    y_pred_enc = fold_model.predict(X_te)
    
    acc = accuracy_score(y_te_enc, y_pred_enc)
    fold_results.append(acc)
    
    print(f"Fold {i}: train={len(X_tr)} rows, test={len(X_te)} rows, "
          f"test period {fold_test['timestamp'].min()} to {fold_test['timestamp'].max()}, "
          f"accuracy={acc:.4f}")

print(f"\nMean accuracy across folds: {np.mean(fold_results):.4f}")
print(f"Std deviation: {np.std(fold_results):.4f}")

Fold 1: train=5343 rows, test=5344 rows, test period 2023-08-12 16:00:00 to 2024-03-22 07:00:00, accuracy=0.6078
Fold 2: train=10687 rows, test=5344 rows, test period 2024-03-22 08:00:00 to 2024-10-30 23:00:00, accuracy=0.6428
Fold 3: train=16031 rows, test=5344 rows, test period 2024-10-31 00:00:00 to 2025-06-10 15:00:00, accuracy=0.6271
Fold 4: train=21375 rows, test=5344 rows, test period 2025-06-10 16:00:00 to 2026-01-19 07:00:00, accuracy=0.6388
Fold 5: train=26719 rows, test=5344 rows, test period 2026-01-19 08:00:00 to 2026-08-29 23:00:00, accuracy=0.6220

Mean accuracy across folds: 0.6277
Std deviation: 0.0125


In [30]:
import joblib

xgb_model_path = "../models/volatility_xgb_final.pkl"
joblib.dump(xgb_model, xgb_model_path)
print(f"Saved final XGBoost model to: {xgb_model_path}")

le_path = "../models/volatility_label_encoder.pkl"
joblib.dump(le, le_path)
print(f"Saved label encoder to: {le_path}")

Saved final XGBoost model to: ../models/volatility_xgb_final.pkl
Saved label encoder to: ../models/volatility_label_encoder.pkl


In [31]:
import joblib
import numpy as np
from xgboost import XGBClassifier

# Prepare complete dataset
X_final = df[feature_cols].replace([np.inf, -np.inf], np.nan)
y_final = df["vol_target"]

# Remove rows with missing features
final_mask = X_final.notnull().all(axis=1)
X_final = X_final[final_mask]
y_final = y_final[final_mask]

# Encode target
final_le = LabelEncoder()
y_final_enc = final_le.fit_transform(y_final)

print("Final training rows:", len(X_final))
print("Features:", len(feature_cols))
print("Classes:", list(final_le.classes_))

# Final XGBoost model
final_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

# Train on ALL usable historical data
final_xgb_model.fit(X_final, y_final_enc)

print("Final XGBoost model trained successfully.")

Final training rows: 32064
Features: 22
Classes: ['CONTRACT', 'EXPAND']
Final XGBoost model trained successfully.


In [32]:
final_model_path = "../models/volatility_xgb_final.pkl"
joblib.dump(final_xgb_model, final_model_path)

final_encoder_path = "../models/volatility_label_encoder.pkl"
joblib.dump(final_le, final_encoder_path)

print(f"Final model saved: {final_model_path}")
print(f"Final label encoder saved: {final_encoder_path}")

Final model saved: ../models/volatility_xgb_final.pkl
Final label encoder saved: ../models/volatility_label_encoder.pkl


In [33]:
import joblib

# Load saved final model
loaded_model = joblib.load("../models/volatility_xgb_final.pkl")
loaded_encoder = joblib.load("../models/volatility_label_encoder.pkl")

print("Model loaded successfully.")
print("Classes:", list(loaded_encoder.classes_))

# Test prediction on the latest available row
latest_X = df[feature_cols].replace([np.inf, -np.inf], np.nan).dropna().tail(1)

prediction_encoded = loaded_model.predict(latest_X)
prediction_label = loaded_encoder.inverse_transform(prediction_encoded)

print("Latest prediction:", prediction_label[0])

Model loaded successfully.
Classes: ['CONTRACT', 'EXPAND']
Latest prediction: EXPAND


In [34]:
def predict_latest_volatility(df, model, encoder, feature_cols):
    """
    Predict the current volatility regime using the latest valid row.
    """

    # Prepare features
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan)

    # Find latest valid row
    valid_rows = X.dropna()

    if valid_rows.empty:
        raise ValueError("No valid feature row available for prediction.")

    latest_X = valid_rows.tail(1)

    # Prediction
    prediction_encoded = model.predict(latest_X)
    prediction_label = encoder.inverse_transform(prediction_encoded)[0]

    # Prediction probabilities
    probabilities = model.predict_proba(latest_X)[0]

    # Map probabilities to class names
    class_probabilities = dict(
        zip(encoder.classes_, probabilities)
    )

    confidence = max(probabilities) * 100

    return {
        "regime": prediction_label,
        "confidence": confidence,
        "probabilities": class_probabilities,
        "timestamp": df.loc[latest_X.index[0], "timestamp"]
    }


# Run prediction
result = predict_latest_volatility(
    df,
    loaded_model,
    loaded_encoder,
    feature_cols
)

print("===================================")
print("      CRYPTO VOLATILITY SIGNAL")
print("===================================")
print(f"Timestamp:  {result['timestamp']}")
print(f"Regime:     {result['regime']}")
print(f"Confidence: {result['confidence']:.2f}%")
print()
print("Class probabilities:")

for cls, prob in result["probabilities"].items():
    print(f"{cls}: {prob * 100:.2f}%")

      CRYPTO VOLATILITY SIGNAL
Timestamp:  2026-08-30 00:00:00
Regime:     EXPAND
Confidence: 50.81%

Class probabilities:
CONTRACT: 49.19%
EXPAND: 50.81%


In [35]:
# Check the latest rows and missing features

check_df = df[["timestamp"] + feature_cols].copy()

check_df = check_df.replace([np.inf, -np.inf], np.nan)

check_df["missing_features"] = check_df[feature_cols].isna().sum(axis=1)

print(check_df.tail(10)[["timestamp", "missing_features"]])

                timestamp  missing_features
32055 2026-08-29 15:00:00                 0
32056 2026-08-29 16:00:00                 0
32057 2026-08-29 17:00:00                 0
32058 2026-08-29 18:00:00                 0
32059 2026-08-29 19:00:00                 0
32060 2026-08-29 20:00:00                 0
32061 2026-08-29 21:00:00                 0
32062 2026-08-29 22:00:00                 0
32063 2026-08-29 23:00:00                 0
32064 2026-08-30 00:00:00                 0


In [36]:
print("\nLatest dataset timestamp:")
print(df["timestamp"].iloc[-1])

print("\nLatest valid feature timestamp:")
print(check_df.dropna(subset=feature_cols)["timestamp"].iloc[-1])


Latest dataset timestamp:
2026-08-30 00:00:00

Latest valid feature timestamp:
2026-08-30 00:00:00


In [37]:
# Get predictions and probabilities on the full dataset

X_check = df[feature_cols].replace([np.inf, -np.inf], np.nan)

valid_mask = X_check.notnull().all(axis=1)

X_check = X_check[valid_mask]
y_check = df.loc[valid_mask, "vol_target"]

pred_encoded = loaded_model.predict(X_check)
pred_proba = loaded_model.predict_proba(X_check)

pred_labels = loaded_encoder.inverse_transform(pred_encoded)

confidence = pred_proba.max(axis=1)

analysis_df = pd.DataFrame({
    "timestamp": df.loc[X_check.index, "timestamp"].values,
    "actual": y_check.values,
    "predicted": pred_labels,
    "confidence": confidence
})

print("Total predictions:", len(analysis_df))

print("\nAverage confidence:")
print(f"{analysis_df['confidence'].mean() * 100:.2f}%")

print("\nConfidence distribution:")
print(
    pd.cut(
        analysis_df["confidence"],
        bins=[0.5, 0.55, 0.60, 0.65, 0.70, 0.80, 1.01],
        labels=[
            "50-55%",
            "55-60%",
            "60-65%",
            "65-70%",
            "70-80%",
            "80%+"
        ]
    ).value_counts().sort_index()
)

Total predictions: 32064

Average confidence:
67.88%

Confidence distribution:
confidence
50-55%    5422
55-60%    5024
60-65%    4575
65-70%    4281
70-80%    6685
80%+      6077
Name: count, dtype: int64


In [38]:
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from xgboost import XGBClassifier

n_splits = 5
total_len = len(df)
fold_size = total_len // (n_splits + 1)

confidence_results = []

for i in range(1, n_splits + 1):

    train_end = fold_size * i
    test_end = fold_size * (i + 1)

    fold_train = df.iloc[:train_end]
    fold_test = df.iloc[train_end:test_end]

    X_tr = fold_train[feature_cols].replace([np.inf, -np.inf], np.nan)
    y_tr = fold_train["vol_target"]

    X_te = fold_test[feature_cols].replace([np.inf, -np.inf], np.nan)
    y_te = fold_test["vol_target"]

    tr_mask = X_tr.notnull().all(axis=1)
    te_mask = X_te.notnull().all(axis=1)

    X_tr = X_tr[tr_mask]
    y_tr = y_tr[tr_mask]

    X_te = X_te[te_mask]
    y_te = y_te[te_mask]

    fold_le = LabelEncoder()

    y_tr_enc = fold_le.fit_transform(y_tr)
    y_te_enc = fold_le.transform(y_te)

    fold_model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss"
    )

    fold_model.fit(X_tr, y_tr_enc)

    # Out-of-sample probabilities
    probabilities = fold_model.predict_proba(X_te)

    predictions = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)

    correct = predictions == y_te_enc

    fold_df = pd.DataFrame({
        "fold": i,
        "confidence": confidence,
        "correct": correct
    })

    confidence_results.append(fold_df)

confidence_df = pd.concat(confidence_results, ignore_index=True)

print("Total out-of-sample predictions:", len(confidence_df))
print()

# Overall accuracy
print(
    "Overall out-of-sample accuracy:",
    f"{confidence_df['correct'].mean() * 100:.2f}%"
)

print("\nAccuracy by confidence:")
print("--------------------------------")

bins = [0.50, 0.55, 0.60, 0.65, 0.70, 0.80, 1.01]
labels = [
    "50-55%",
    "55-60%",
    "60-65%",
    "65-70%",
    "70-80%",
    "80%+"
]

confidence_df["confidence_band"] = pd.cut(
    confidence_df["confidence"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

summary = (
    confidence_df
    .groupby("confidence_band", observed=False)
    .agg(
        predictions=("correct", "count"),
        accuracy=("correct", "mean"),
        avg_confidence=("confidence", "mean")
    )
)

summary["accuracy"] *= 100
summary["avg_confidence"] *= 100

print(summary.round(2))

Total out-of-sample predictions: 26720

Overall out-of-sample accuracy: 62.77%

Accuracy by confidence:
--------------------------------
                 predictions  accuracy  avg_confidence
confidence_band                                       
50-55%                  3329     51.19       52.500000
55-60%                  3410     54.34       57.520000
60-65%                  3285     55.74       62.459999
65-70%                  3065     60.65       67.489998
70-80%                  5875     64.26       74.900002
80%+                    7756     74.14       87.720001


In [39]:
def get_volatility_signal(df, model, encoder, feature_cols):

    X = df[feature_cols].replace([np.inf, -np.inf], np.nan)

    valid_rows = X.dropna()

    if valid_rows.empty:
        raise ValueError("No valid feature row available.")

    latest_X = valid_rows.tail(1)
    latest_index = latest_X.index[0]

    # Prediction
    probabilities = model.predict_proba(latest_X)[0]

    predicted_index = probabilities.argmax()
    regime = encoder.inverse_transform([predicted_index])[0]

    confidence = probabilities[predicted_index] * 100

    # Confidence classification
    if confidence < 60:
        signal_strength = "NO SIGNAL"
    elif confidence < 70:
        signal_strength = "LOW"
    elif confidence < 80:
        signal_strength = "MODERATE"
    else:
        signal_strength = "HIGH"

    return {
        "timestamp": df.loc[latest_index, "timestamp"],
        "regime": regime,
        "confidence": confidence,
        "signal_strength": signal_strength,
        "contract_probability": probabilities[
            list(encoder.classes_).index("CONTRACT")
        ] * 100,
        "expand_probability": probabilities[
            list(encoder.classes_).index("EXPAND")
        ] * 100
    }


signal = get_volatility_signal(
    df,
    loaded_model,
    loaded_encoder,
    feature_cols
)

print("===================================")
print("       VOLATILITY ML SIGNAL")
print("===================================")
print(f"Timestamp:            {signal['timestamp']}")
print(f"Regime:               {signal['regime']}")
print(f"Model confidence:     {signal['confidence']:.2f}%")
print(f"Signal strength:      {signal['signal_strength']}")
print()
print(f"CONTRACT probability: {signal['contract_probability']:.2f}%")
print(f"EXPAND probability:   {signal['expand_probability']:.2f}%")

       VOLATILITY ML SIGNAL
Timestamp:            2026-08-30 00:00:00
Regime:               EXPAND
Model confidence:     50.81%
Signal strength:      NO SIGNAL

CONTRACT probability: 49.19%
EXPAND probability:   50.81%


In [40]:
print(confidence_df.head())
print()
print(confidence_df.columns.tolist())

   fold  confidence  correct confidence_band
0     1    0.706602     True          70-80%
1     1    0.832302     True            80%+
2     1    0.853044     True            80%+
3     1    0.811998     True            80%+
4     1    0.865125     True            80%+

['fold', 'confidence', 'correct', 'confidence_band']


In [41]:
print("DataFrame columns:")
print(df.columns.tolist())

print("\nVolatility target distribution:")
print(df["vol_target"].value_counts())

print("\nVolatility target percentages:")
print(df["vol_target"].value_counts(normalize=True).mul(100).round(2))

DataFrame columns:
['timestamp', 'open', 'high', 'low', 'close', 'volume', 'return_1h', 'return_3h', 'return_6h', 'return_24h', 'ema20', 'ema50', 'ema200', 'ema20_dist', 'ema50_dist', 'ema200_dist', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_middle', 'bb_upper', 'bb_lower', 'bb_width', 'atr', 'volume_change', 'volume_ratio', 'volatility_24h', 'future_return_6h', 'target', 'target_binary', 'future_return_24h', 'target_24h', 'future_volatility', 'vol_target']

Volatility target distribution:
vol_target
EXPAND      16038
CONTRACT    16027
Name: count, dtype: int64

Volatility target percentages:
vol_target
EXPAND      50.02
CONTRACT    49.98
Name: proportion, dtype: float64


In [42]:
print("\nLatest rows:")
print(
    df[["timestamp", "close", "vol_target"]]
    .tail(10)
)


Latest rows:
                timestamp     close vol_target
32055 2026-08-29 15:00:00  77865.36   CONTRACT
32056 2026-08-29 16:00:00  78027.17   CONTRACT
32057 2026-08-29 17:00:00  78100.00   CONTRACT
32058 2026-08-29 18:00:00  78158.01   CONTRACT
32059 2026-08-29 19:00:00  78182.00   CONTRACT
32060 2026-08-29 20:00:00  78129.92   CONTRACT
32061 2026-08-29 21:00:00  78161.63   CONTRACT
32062 2026-08-29 22:00:00  78228.34   CONTRACT
32063 2026-08-29 23:00:00  78230.00   CONTRACT
32064 2026-08-30 00:00:00  78179.86   CONTRACT


In [43]:
print("Volatility target distribution:")
print(df["vol_target"].value_counts())

print("\nVolatility target percentages:")
print(
    df["vol_target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Volatility target distribution:
vol_target
EXPAND      16038
CONTRACT    16027
Name: count, dtype: int64

Volatility target percentages:
vol_target
EXPAND      50.02
CONTRACT    49.98
Name: proportion, dtype: float64


In [44]:
print(df[["timestamp", "close", "volatility_24h", "vol_target"]].tail(30))

                timestamp     close  volatility_24h vol_target
32035 2026-08-28 19:00:00  77580.03        0.004200   CONTRACT
32036 2026-08-28 20:00:00  77407.34        0.004135   CONTRACT
32037 2026-08-28 21:00:00  77364.00        0.004105   CONTRACT
32038 2026-08-28 22:00:00  77758.69        0.004282   CONTRACT
32039 2026-08-28 23:00:00  77845.87        0.004309   CONTRACT
32040 2026-08-29 00:00:00  77733.63        0.004157   CONTRACT
32041 2026-08-29 01:00:00  77794.11        0.004182   CONTRACT
32042 2026-08-29 02:00:00  77671.18        0.004001   CONTRACT
32043 2026-08-29 03:00:00  77507.17        0.004006   CONTRACT
32044 2026-08-29 04:00:00  77648.91        0.004048   CONTRACT
32045 2026-08-29 05:00:00  77632.53        0.004038   CONTRACT
32046 2026-08-29 06:00:00  77466.82        0.003960   CONTRACT
32047 2026-08-29 07:00:00  77626.01        0.003997   CONTRACT
32048 2026-08-29 08:00:00  77633.99        0.004001   CONTRACT
32049 2026-08-29 09:00:00  77650.03        0.003987   C

In [45]:
import joblib

feature_cols_path = "../models/volatility_xgb_final_features.pkl"

joblib.dump(feature_cols, feature_cols_path)

print(f"Final feature list saved: {feature_cols_path}")
print(f"Number of features: {len(feature_cols)}")
print("\nFeatures:")
for i, feature in enumerate(feature_cols, 1):
    print(f"{i}. {feature}")

Final feature list saved: ../models/volatility_xgb_final_features.pkl
Number of features: 22

Features:
1. return_1h
2. return_3h
3. return_6h
4. return_24h
5. ema20
6. ema50
7. ema200
8. ema20_dist
9. ema50_dist
10. ema200_dist
11. rsi
12. macd
13. macd_signal
14. macd_hist
15. bb_middle
16. bb_upper
17. bb_lower
18. bb_width
19. atr
20. volume_change
21. volume_ratio
22. volatility_24h


In [46]:
import joblib

model_metadata = {
    "model_type": "XGBoost",
    "target": "vol_target",
    "prediction_horizon_hours": 6,
    "classes": list(final_le.classes_),
    "n_features": len(feature_cols),
    "features": feature_cols,
    "n_estimators": 300,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "walk_forward_folds": 5,
    "walk_forward_mean_accuracy": 0.6277,
    "walk_forward_std": 0.0125,
    "high_confidence_threshold": 0.80
}

metadata_path = "../models/volatility_xgb_metadata.pkl"

joblib.dump(model_metadata, metadata_path)

print(f"Metadata saved: {metadata_path}")
print("\nModel metadata:")
for key, value in model_metadata.items():
    print(f"{key}: {value}")

Metadata saved: ../models/volatility_xgb_metadata.pkl

Model metadata:
model_type: XGBoost
target: vol_target
prediction_horizon_hours: 6
classes: ['CONTRACT', 'EXPAND']
n_features: 22
features: ['return_1h', 'return_3h', 'return_6h', 'return_24h', 'ema20', 'ema50', 'ema200', 'ema20_dist', 'ema50_dist', 'ema200_dist', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_middle', 'bb_upper', 'bb_lower', 'bb_width', 'atr', 'volume_change', 'volume_ratio', 'volatility_24h']
n_estimators: 300
max_depth: 5
learning_rate: 0.05
subsample: 0.8
colsample_bytree: 0.8
walk_forward_folds: 5
walk_forward_mean_accuracy: 0.6277
walk_forward_std: 0.0125
high_confidence_threshold: 0.8


In [47]:
import os
import joblib

model_files = [
    "../models/volatility_xgb_final.pkl",
    "../models/volatility_xgb_final_features.pkl",
    "../models/volatility_label_encoder.pkl",
    "../models/volatility_xgb_metadata.pkl"
]

print("===================================")
print("       MODEL PACKAGE CHECK")
print("===================================")

all_present = True

for path in model_files:
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    print(f"{status:8} {path}")
    
    if not exists:
        all_present = False

print()

if all_present:
    model = joblib.load("../models/volatility_xgb_final.pkl")
    features = joblib.load("../models/volatility_xgb_final_features.pkl")
    encoder = joblib.load("../models/volatility_label_encoder.pkl")
    metadata = joblib.load("../models/volatility_xgb_metadata.pkl")

    print("All model files loaded successfully.")
    print(f"Model type: {metadata['model_type']}")
    print(f"Features: {len(features)}")
    print(f"Classes: {list(encoder.classes_)}")
    print(f"Prediction horizon: {metadata['prediction_horizon_hours']} hours")
    print(f"Walk-forward accuracy: {metadata['walk_forward_mean_accuracy'] * 100:.2f}%")
    print(f"Walk-forward std: {metadata['walk_forward_std'] * 100:.2f}%")
else:
    print("ERROR: One or more model files are missing.")

       MODEL PACKAGE CHECK
OK       ../models/volatility_xgb_final.pkl
OK       ../models/volatility_xgb_final_features.pkl
OK       ../models/volatility_label_encoder.pkl
OK       ../models/volatility_xgb_metadata.pkl

All model files loaded successfully.
Model type: XGBoost
Features: 22
Classes: ['CONTRACT', 'EXPAND']
Prediction horizon: 6 hours
Walk-forward accuracy: 62.77%
Walk-forward std: 1.25%
